In [3]:
import os, glob, shutil, tempfile, time, subprocess
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from IPython.display import HTML, display
from openpyxl import load_workbook

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
TARGET_DATE  = '2026-08-24'
SEND_EMAIL   = False

BASE_HC    = os.path.join(os.path.expanduser('~'), 'Concentrix Corporation',
                          'WFM-Expedia-HCM - Branding files', 'Headcount')
HC_EXT_DIR = os.path.join(BASE_HC, 'HC Extend by Month')
HC_MASTER  = os.path.join(BASE_HC, 'HC Master Database - 2026.xlsx')
WD_DIR     = os.path.join(BASE_HC, 'WD')
TEMP_DIR   = tempfile.mkdtemp()

EMAIL_TO = (
    'Van Tran <van.tran@concentrix.com>; '
    'ASEAN Reporting <ASEAN_Reporting@concentrix.com>; '
    'Puneet Suneja <puneet.suneja@concentrix.com>; '
    'KIRPAN PATAR <kirpan.patar@concentrix.com>'
)
EMAIL_CC = (
    'Varun Kathuria <Varun.Kathuria@concentrix.com>; '
    'Urmila Chakka <urmila.chakka1@concentrix.com>; '
    'Pagarigan Silastre Aimee <aimee.silastre@concentrix.com>; '
    'Rajat Roy <rajat.roy@concentrix.com>; '
    'Lokesh Yadav <lokesh.yadav2@concentrix.com>; '
    'Abishek . <abishek.a@concentrix.com>; '
    'My Duyen Ly <myduyen.ly@concentrix.com>; '
    'Dien Bui <huongdien.bui@concentrix.com>; '
    'VN_HCM_ONE_EXP_WFM <VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com>'
)

dt            = datetime.strptime(TARGET_DATE, '%Y-%m-%d')
WB_LABEL      = 'WB' + dt.strftime('%y%m%d')
DATE_DISPLAY  = dt.strftime('%m.%d.%Y')
CUR_MONTH     = dt.strftime('%Y_%m')
TARGET_TS     = pd.Timestamp(TARGET_DATE)
EMAIL_SUBJECT = f'Expedia VN - Headcount Report as of {DATE_DISPLAY}'

TQG_DESIG  = {'Trainer II', 'Supervisor, Training & Quality',
              'Sr. Quality Evaluator', 'Quality Evaluator'}
OPS_DESIG  = {'Advisor I, Customer Service', 'Operations Manager I',
              'Sr. SME, Operations', 'SME, Operations', 'Team Leader, Operations'}
WFM_DESIG  = {'Analyst, WFM Real Time Management',
              'Sr. Representative, WFM Real Time Management',
              'Sr. Representative, Real Time Management'}
FUNC_ORDER = ['Operations Group', 'Training & Quality Group', 'WFM Group']
DROP_COLS  = ['Mini TL - Email', 'Mini TL - Short Name', 'Mini TL Start Date']
CATS       = ['Retail Agent', 'HPO Agent', 'TQA', 'Ops Support', 'WFM', 'Terminated']

def to_week_monday(ts):
    """Map any date to its week's Monday (start of week label)."""
    if pd.isna(ts): return None
    return (ts - pd.Timedelta(days=ts.weekday())).date()

_cur_mon           = to_week_monday(TARGET_TS)
last_closed_monday = _cur_mon
EIGHT_WEEKS = {_cur_mon - timedelta(weeks=i) for i in range(8)}

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def find_col(df, name):
    return next((c for c in df.columns if c.strip().lower() == name.strip().lower()), None)

def load_sheet_safe(path, sheet_name):
    tmp = os.path.join(TEMP_DIR, f'_tmp_{os.path.basename(path)}')
    shutil.copy2(path, tmp)
    xl = pd.ExcelFile(tmp, engine='openpyxl')
    names = xl.sheet_names; xl.close()
    actual = next((s for s in names if s.strip().lower() == sheet_name.strip().lower()), None)
    if actual is None:
        return pd.DataFrame()
    df = pd.read_excel(tmp, sheet_name=actual, dtype=str, engine='openpyxl')
    for _ in range(5):
        try: os.remove(tmp); break
        except PermissionError: time.sleep(0.3)
    return df

def get_function(desig):
    d = str(desig).strip() if pd.notna(desig) else ''
    if d in TQG_DESIG: return 'Training & Quality Group'
    if d in OPS_DESIG: return 'Operations Group'
    if d in WFM_DESIG: return 'WFM Group'
    return 'Other'

_RETAIL_LOB_STARTS = (
    'LG ',          # LG Chat, LG Chat CSG, LG Chat CSG Training, LG Nesting Chat
    'LG_',          # LG_...
    'NL ',          # NL Chat
    'NL_',
    'Lodging',      # Lodging_Training
    'Non_Lodging', 'Non Lodging', 'Non-Lodging',
    'Support_LG',   # Support_LG_Nesting
    'Support - LG', # Support - LG
    'Support - NL', # Support - NL (Non-Lodging support)
)
_TQA_LOB_EXACT = {'TQA', 'Support_Flex_QA', 'Support_Flex_Trainer'}
_OPS_LOB_EXACT = {'Support_Flex_SME'}
_WFM_LOB_EXACT = {'WFM'}

def _assign_atr_cat(lob, desig, status):
    l = str(lob).strip()    if pd.notna(lob)    else ''
    d = str(desig).strip()  if pd.notna(desig)  else ''
    s = str(status).strip() if pd.notna(status) else ''
    if s in ['Terminated', 'Transferred']:
        return 'Terminated'
    if any(l.startswith(p) for p in _RETAIL_LOB_STARTS):
        return 'Retail Agent'
    if l.startswith('HPO'):
        return 'HPO Agent'
    if l in _WFM_LOB_EXACT:
        return 'WFM'
    if l in _TQA_LOB_EXACT:
        return 'TQA'
    if l in _OPS_LOB_EXACT:
        return 'Ops Support'
    f = get_function(d)
    if f == 'Training & Quality Group': return 'TQA'
    if f == 'Operations Group':         return 'Ops Support'
    if f == 'WFM Group':                return 'WFM'
    return 'Ops Support'

# ══════════════════════════════════════════════════════════════════════════════
# LOAD ALL WD TERMINATIONS (all months, all Termination sheets)
# ══════════════════════════════════════════════════════════════════════════════
def load_all_wd_terminations():
    parts = []
    for wdf in sorted(glob.glob(os.path.join(WD_DIR, '*.xlsx'))):
        df_t = load_sheet_safe(wdf, 'Termination')
        if not df_t.empty:
            parts.append(df_t)
    if not parts:
        print('  WD Termination: no sheets found'); return pd.DataFrame(), None, None

    df_all = pd.concat(parts, ignore_index=True)
    tc     = find_col(df_all, 'Termination Date')
    lwd_c  = find_col(df_all, 'LWD')
    ec     = find_col(df_all, 'EMPLOYEE_NUMBER') or find_col(df_all, 'OracleID')

    if tc:   df_all[tc]    = pd.to_datetime(df_all[tc],    errors='coerce')
    if lwd_c:df_all[lwd_c] = pd.to_datetime(df_all[lwd_c], errors='coerce')
    if tc and lwd_c:
        df_all['_eff'] = df_all[tc].fillna(df_all[lwd_c]); date_col = '_eff'
    elif tc:  date_col = tc
    elif lwd_c: date_col = lwd_c
    else: print('  WD: date column not found'); return pd.DataFrame(), None, None

    if ec:
        df_all[ec] = pd.to_numeric(df_all[ec], errors='coerce')
        df_all = df_all.dropna(subset=[ec, date_col])
        df_all = df_all.sort_values(date_col).drop_duplicates(subset=[ec], keep='first')
    else:
        df_all = df_all.dropna(subset=[date_col])

    print(f'  WD Terminated unique: {len(df_all)} | date={date_col} | emp={ec}')
    return df_all, date_col, ec

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: Load & combine HC Extend CSVs
# ══════════════════════════════════════════════════════════════════════════════
print('Loading HC Extend CSVs...')
csv_files = glob.glob(os.path.join(HC_EXT_DIR, '*.csv'))
if not csv_files: raise FileNotFoundError(f'No CSV in: {HC_EXT_DIR}')

dfs = []
for f in csv_files:
    try: dfs.append(pd.read_csv(f, dtype=str, encoding='utf-8-sig'))
    except Exception as e: print(f'  Skipped {os.path.basename(f)}: {e}')
df_hc = pd.concat(dfs, ignore_index=True)
print(f'  Combined: {len(df_hc)} rows')

df_hc_all = df_hc.copy()
_dsw_a = find_col(df_hc_all, 'Date Start Week')
_sts_a = find_col(df_hc_all, 'Detail Status')
_dsg_a = find_col(df_hc_all, 'Designation')
_ora_a = find_col(df_hc_all, 'OracleID')
_lob_a = (find_col(df_hc_all, 'LOB_3') or
          find_col(df_hc_all, 'LOB') or
          find_col(df_hc_all, 'LOB_Combine'))
if _dsw_a: df_hc_all[_dsw_a] = pd.to_datetime(df_hc_all[_dsw_a], errors='coerce')
print(f'  Attrition base: {len(df_hc_all)} rows | LOB col: {_lob_a}')
if _lob_a and _lob_a in df_hc_all.columns:
    print(f'  LOB unique vals: {sorted(df_hc_all[_lob_a].dropna().unique().tolist())}')
dsw_col    = find_col(df_hc, 'Date Start Week')
day_col    = find_col(df_hc, 'Day')
status_col = find_col(df_hc, 'Detail Status')
oracle_col = find_col(df_hc, 'OracleID')
desig_col  = find_col(df_hc, 'Designation')
lob3_col   = find_col(df_hc, 'LOB_3') or find_col(df_hc, 'LOB')

if dsw_col:
    df_hc[dsw_col] = pd.to_datetime(df_hc[dsw_col], errors='coerce')
    df_hc = df_hc[df_hc[dsw_col].dt.strftime('%Y-%m-%d') == TARGET_DATE]
    df_hc[dsw_col] = df_hc[dsw_col].dt.date
if day_col:    df_hc = df_hc[df_hc[day_col].str.strip() == 'Mon']
if status_col: df_hc = df_hc[~df_hc[status_col].str.strip().isin(['Terminated', 'Transferred'])]
if lob3_col:   df_hc = df_hc[df_hc[lob3_col].fillna('').str.strip().apply(lambda x: not x.startswith('HPO'))]
for col in DROP_COLS:
    actual = find_col(df_hc, col)
    if actual: df_hc.drop(columns=[actual], inplace=True)
if desig_col:
    pos = df_hc.columns.get_loc(desig_col) + 1
    df_hc.insert(pos, 'Function', df_hc[desig_col].apply(get_function))
df_hc = df_hc.reset_index(drop=True)
print(f'  Target week rows: {len(df_hc)}')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: EWS
# ══════════════════════════════════════════════════════════════════════════════
print('Loading EWS...')
df_ews_raw = load_sheet_safe(HC_MASTER, 'EWS')
ews_ora = find_col(df_ews_raw, 'OracleID'); ews_lwd = find_col(df_ews_raw, 'LWD Expected')
ews_iex = find_col(df_ews_raw, 'IEX ID');  ews_emp = find_col(df_ews_raw, 'Employee Name')
ews_lob = find_col(df_ews_raw, 'LOB');     ews_sup = find_col(df_ews_raw, 'Supervisor Name')
df_ews_out = pd.DataFrame()
if ews_ora and ews_lwd:
    df_ews_raw[ews_ora] = pd.to_numeric(df_ews_raw[ews_ora], errors='coerce')
    df_ews_raw[ews_lwd] = pd.to_datetime(df_ews_raw[ews_lwd], errors='coerce')
    ews_lkp = (df_ews_raw.dropna(subset=[ews_ora, ews_lwd]).drop_duplicates(ews_ora)
               .set_index(ews_ora)[ews_lwd].to_dict())
    if oracle_col:
        df_hc[oracle_col] = pd.to_numeric(df_hc[oracle_col], errors='coerce')
        df_hc['EWS'] = df_hc[oracle_col].map(ews_lkp).apply(
            lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else '-')
    df_ews_f = df_ews_raw[df_ews_raw[ews_lwd] >= TARGET_TS].copy()
    if ews_lob: df_ews_f = df_ews_f[~df_ews_f[ews_lob].fillna('').str.strip().apply(lambda x: x.startswith('HPO'))]
    df_ews_f = df_ews_f.sort_values(ews_lwd)
    if ews_iex: df_ews_f[ews_iex] = pd.to_numeric(df_ews_f[ews_iex], errors='coerce').astype('Int64')
    df_ews_f[ews_lwd] = df_ews_f[ews_lwd].dt.strftime('%Y-%m-%d')
    df_ews_out = df_ews_f[[c for c in [ews_iex, ews_emp, ews_lob, ews_sup, ews_lwd] if c]].reset_index(drop=True)
    print(f'  EWS display: {len(df_ews_out)} rows')
else:
    df_hc['EWS'] = '-'

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: Active column
# ══════════════════════════════════════════════════════════════════════════════
print('Loading WD current month...')
wd_files = (glob.glob(os.path.join(WD_DIR, f'WD_{CUR_MONTH}*.xlsx')) or
            glob.glob(os.path.join(WD_DIR, f'*{CUR_MONTH}*.xlsx')) or
            sorted(glob.glob(os.path.join(WD_DIR, '*.xlsx')), key=os.path.getmtime, reverse=True))
wd_file_path = wd_files[0] if wd_files else None
print(f'  WD current: {os.path.basename(wd_file_path) if wd_file_path else "NOT FOUND"}')

terminated_set = set()
if wd_file_path:
    df_term_cur = load_sheet_safe(wd_file_path, 'Termination')
    if not df_term_cur.empty:
        tc_cur = find_col(df_term_cur, 'EMPLOYEE_NUMBER') or find_col(df_term_cur, 'OracleID')
        if tc_cur: terminated_set = set(pd.to_numeric(df_term_cur[tc_cur], errors='coerce').dropna())
    else:
        xl_tmp = os.path.join(TEMP_DIR, '_wd_main.xlsx')
        shutil.copy2(wd_file_path, xl_tmp)
        xl = pd.ExcelFile(xl_tmp, engine='openpyxl')
        df_wd = pd.read_excel(xl_tmp, sheet_name=xl.sheet_names[0], dtype=str, engine='openpyxl')
        xl.close()
        for _ in range(5):
            try: os.remove(xl_tmp); break
            except PermissionError: time.sleep(0.3)
        ec2 = find_col(df_wd, 'EMPLOYEE_NUMBER'); sc2 = find_col(df_wd, 'Employee Status')
        if ec2 and sc2:
            terminated_set = set(pd.to_numeric(
                df_wd[df_wd[sc2].str.strip() == 'INACTIVE'][ec2], errors='coerce').dropna())

if oracle_col:
    is_term = df_hc[oracle_col].isin(terminated_set)
    has_ews = df_hc['EWS'] != '-'
    df_hc['Active'] = 'Active'
    df_hc.loc[has_ews & ~is_term, 'Active'] = 'Serving notice'
    df_hc.loc[is_term, 'Active'] = 'Inactive'
else:
    df_hc['Active'] = 'Active'

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: File 1
# ══════════════════════════════════════════════════════════════════════════════
df_inactive = load_sheet_safe(HC_MASTER, 'Inactive')
file1_name  = f'HC Master Database - {WB_LABEL}.xlsx'
file1_path  = os.path.join(TEMP_DIR, file1_name)
with pd.ExcelWriter(file1_path, engine='openpyxl', date_format='YYYY-MM-DD') as writer:
    df_hc.to_excel(writer, index=False, sheet_name='Active')
    df_inactive.to_excel(writer, index=False, sheet_name='Inactive')
wb1 = load_workbook(file1_path); ws1 = wb1['Active']
hdr = {cell.value: cell.column for cell in ws1[1] if cell.value}
if dsw_col and dsw_col in hdr:
    for r in range(2, ws1.max_row + 1):
        ws1.cell(r, hdr[dsw_col]).number_format = 'YYYY-MM-DD'
wb1.save(file1_path); wb1.close()
file2_path = wd_file_path
file2_name = os.path.basename(wd_file_path) if wd_file_path else 'WD not found'
print(f'File 1: {file1_name} | File 2: {file2_name}')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5: Load ALL WD Termination data (for attrition count)
# ══════════════════════════════════════════════════════════════════════════════
print('\nLoading all WD Termination sheets...')
df_wd_all_term, wd_tc, wd_ec = load_all_wd_terminations()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6: HTML BUILDERS
# ══════════════════════════════════════════════════════════════════════════════
FONT    = 'font-family:Calibri,Arial,sans-serif;font-size:11px;'
D_DARK  = '#1F3864'
D_YELL  = '#FFFF00'
FUNC_BG = '#D6E4F0'
ATR_HDR = '#3A6D9E'
ROW_EVEN= '#EBF5FB'
WHITE   = '#FFFFFF'
TH = f'padding:4px 6px;border:1px solid #fff;background:{ATR_HDR};color:#fff;text-align:center;{FONT}font-weight:bold;'
GT = f'padding:4px 6px;border:1px solid #fff;background:{D_DARK};color:#fff;text-align:center;{FONT}font-weight:bold;'
TD = f'padding:3px 5px;border:1px solid #D5D8DC;text-align:center;{FONT}'

def build_hc_pivot_html(df, lob_col_name):
    if not lob_col_name or lob_col_name not in df.columns: return '<p>LOB column not found</p>'
    df_g = df.groupby(['Function','Designation','Active', lob_col_name]).size().reset_index(name='n')
    pvt  = df_g.pivot_table(index=['Function','Designation','Active'],
                            columns=lob_col_name, values='n', aggfunc='sum', fill_value=0)
    pvt.columns.name = None; pvt['Grand Total'] = pvt.sum(axis=1)
    lob_cols = [c for c in pvt.columns if c != 'Grand Total'] + ['Grand Total']
    pvt = pvt[lob_cols]; grand = pvt.sum()
    h = f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 4px;"><thead><tr>'
    h += f'<th style="{TH}text-align:left;">Function</th>'
    h += f'<th style="padding:4px 6px;border:1px solid #fff;background:{D_YELL};color:#FF0000;text-align:center;{FONT}font-weight:bold;">Designation</th>'
    h += f'<th style="{TH}">Status</th>'
    for c in lob_cols: h += f'<th style="{TH}">{c}</th>'
    h += '</tr></thead><tbody>'
    func_seen = [f for f in FUNC_ORDER if f in [i[0] for i in pvt.index]]
    extra_f   = [f for f in dict.fromkeys(i[0] for i in pvt.index) if f not in FUNC_ORDER]
    for func in func_seen + extra_f:
        rows_f = [(d,s,r) for (f,d,s),r in pvt.iterrows() if f == func]
        if not rows_f: continue
        d_dict = {}
        for d,s,r in rows_f: d_dict.setdefault(d,{})[s] = r
        fspan = sum(len(v) for v in d_dict.values()); ff = True
        for desig, s_dict in d_dict.items():
            dspan = len(s_dict); fd = True
            for status, row in s_dict.items():
                h += '<tr>'
                if ff:
                    h += f'<td rowspan="{fspan}" style="{TD}background:{FUNC_BG};font-weight:bold;text-align:left;vertical-align:middle;">{func}</td>'
                    ff = False
                if fd:
                    h += f'<td rowspan="{dspan}" style="{TD}background:{WHITE};vertical-align:middle;">{desig}</td>'
                    fd = False
                h += f'<td style="{TD}background:{WHITE};">{status}</td>'
                for c in lob_cols:
                    v  = int(row[c]) if row[c] != 0 else ''
                    fw = 'bold' if c=='Grand Total' else 'normal'
                    h += f'<td style="{TD}background:{WHITE};font-weight:{fw};">{v}</td>'
                h += '</tr>'
    h += f'<tr><td colspan="3" style="{GT}">Grand Total</td>'
    for c in lob_cols: h += f'<td style="{GT}">{int(grand[c])}</td>'
    return h + '</tr></tbody></table>'

def build_ews_total_html(df_ews, sup_col, lob_col, iex_col_name):
    sc=find_col(df_ews,sup_col); lc=find_col(df_ews,lob_col); ic=find_col(df_ews,iex_col_name)
    if not(sc and lc and ic) or df_ews.empty:
        return f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 4px;"><thead><tr><th style="{TH}">No EWS data</th></tr></thead></table>'
    pvt = df_ews.pivot_table(index=sc, columns=lc, values=ic, aggfunc='count', fill_value=0)
    pvt.columns.name = None; pvt['Grand Total'] = pvt.sum(axis=1)
    lob_cols = [c for c in pvt.columns if c != 'Grand Total'] + ['Grand Total']
    pvt = pvt[lob_cols]; grand = pvt.sum()
    h  = f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 4px;"><thead><tr><th style="{TH}text-align:left;">Supervisor Name</th>'
    for c in lob_cols: h += f'<th style="{TH}">{c}</th>'
    h += '</tr></thead><tbody>'
    for i,(sup,row) in enumerate(pvt.iterrows()):
        bg = ROW_EVEN if i%2==0 else WHITE
        td = f'padding:3px 5px;border:1px solid #D5D8DC;background:{bg};text-align:center;{FONT}'
        h += f'<tr><td style="{td}text-align:left;">{sup}</td>'
        for c in lob_cols:
            fw = 'bold' if c=='Grand Total' else 'normal'
            h += f'<td style="{td}font-weight:{fw};">{int(row[c]) if row[c]!=0 else ""}</td>'
        h += '</tr>'
    h += f'<tr><td style="{GT}text-align:left;">Grand Total</td>'
    for c in lob_cols: h += f'<td style="{GT}">{int(grand[c])}</td>'
    return h + '</tr></tbody></table>'

def build_ews_list_html(df_ews):
    if df_ews.empty:
        return f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 4px;"><thead><tr><th style="{TH}">No EWS records</th></tr></thead></table>'
    cols = list(df_ews.columns)
    h  = f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 4px;"><thead><tr>'
    for c in cols: h += f'<th style="{TH}">{c}</th>'
    h += '</tr></thead><tbody>'
    for i,(_,row) in enumerate(df_ews.iterrows()):
        bg = ROW_EVEN if i%2==0 else WHITE
        td = f'padding:3px 5px;border:1px solid #D5D8DC;background:{bg};text-align:center;{FONT}'
        h += '<tr>'
        for c in cols:
            v = str(row[c]) if pd.notna(row[c]) and str(row[c]) not in ('','nan','<NA>') else ''
            h += f'<td style="{td}">{v}</td>'
        h += '</tr>'
    return h + '</tbody></table>'

def _pivot_attrition(df_src, col_fn, filter_periods=None):
    if not (_dsw_a and _dsg_a and _sts_a) or _dsw_a not in df_src.columns:
        return pd.DataFrame()
    df = df_src.copy()
    df['_col'] = df[_dsw_a].apply(lambda x: col_fn(x) if pd.notna(x) else None)
    df = df.dropna(subset=['_col'])
    if df.empty: return pd.DataFrame()
    df = df.sort_values(_dsw_a)

    _is_term = (df[_sts_a].fillna('').str.strip().isin(['Terminated','Transferred'])
                if _sts_a in df.columns else pd.Series(False, index=df.index))
    df_non = df[~_is_term].copy()

    _period_last = (df_non.groupby('_col')[_dsw_a].max()
                    .reset_index().rename(columns={_dsw_a: '_max_date'}))
    df_act = (df_non
              .merge(_period_last, on='_col', how='left')
              .loc[lambda x: x[_dsw_a] == x['_max_date']]
              .drop(columns=['_max_date']))
    if _ora_a and _ora_a in df_act.columns:
        df_act = df_act.drop_duplicates(subset=[_ora_a, '_col'])

    l_v = df_act[_lob_a].values if (_lob_a and _lob_a in df_act.columns) else ['']*len(df_act)
    d_v = df_act[_dsg_a].values if (_dsg_a and _dsg_a in df_act.columns) else ['']*len(df_act)
    s_v = df_act[_sts_a].values if (_sts_a and _sts_a in df_act.columns) else ['']*len(df_act)
    df_act['_cat'] = [_assign_atr_cat(l,d,s) for l,d,s in zip(l_v, d_v, s_v)]
    if filter_periods is not None:
        df_act = df_act[df_act['_col'].isin(filter_periods)]
    if not df_wd_all_term.empty and wd_ec and wd_tc and _ora_a and _ora_a in df_act.columns:
        df_wd_t = df_wd_all_term[[wd_ec, wd_tc]].copy()
        df_wd_t['_term_col'] = df_wd_t[wd_tc].apply(
            lambda x: col_fn(x) if pd.notna(x) else None)
        df_wd_t = (df_wd_t.dropna(subset=['_term_col'])
                   .rename(columns={wd_ec: _ora_a}))
        df_act = df_act.merge(
            df_wd_t[[_ora_a, '_term_col']], on=_ora_a, how='left')
        df_act = df_act[
            df_act['_term_col'].isna() |
            (df_act['_term_col'] > df_act['_col'])
        ]
        df_act = df_act.drop(columns=['_term_col'])
    if not df_wd_all_term.empty and wd_tc:
        df_t = df_wd_all_term.copy()
        df_t['_col'] = df_t[wd_tc].apply(lambda x: col_fn(x) if pd.notna(x) else None)
        df_t = df_t.dropna(subset=['_col'])
        if wd_ec and wd_ec in df_t.columns:
            df_t = df_t.drop_duplicates(subset=[wd_ec, '_col'])  # unique OracleID per period
        if filter_periods is not None:
            df_t = df_t[df_t['_col'].isin(filter_periods)]
        ft = pd.DataFrame({'_col': df_t['_col'], '_cat': 'Terminated'})
    else:
        df_trows = df[_is_term].copy()
        if not df_trows.empty and _ora_a and _ora_a in df_trows.columns:
            fft = df_trows.groupby(_ora_a)['_col'].min().reset_index()
            fft.columns = [_ora_a,'_col']; fft['_cat'] = 'Terminated'
            if filter_periods is not None: fft = fft[fft['_col'].isin(filter_periods)]
            ft = fft[['_col','_cat']]
        else:
            ft = pd.DataFrame(columns=['_col','_cat'])

    rows = pd.concat([df_act[['_cat','_col']], ft], ignore_index=True)
    if rows.empty: return pd.DataFrame()

    pvt = rows.groupby(['_cat','_col']).size().unstack('_col', fill_value=0)
    pvt = pvt[sorted(pvt.columns)]
    for c in CATS:
        if c not in pvt.index: pvt.loc[c] = 0
    pvt = pvt.loc[CATS]
    active_sum = pvt.drop(index='Terminated', errors='ignore').sum()
    term_row   = pvt.loc['Terminated'] if 'Terminated' in pvt.index else pd.Series(0, index=pvt.columns)
    pvt.loc['% Attrition'] = (term_row / active_sum.replace(0, np.nan) * 100).round(1)
    return pvt

def build_attrition_html(pvt, col_fmt='date'):
    if pvt is None or pvt.empty: return '<p>No attrition data</p>'
    def fmt(c):
        if col_fmt=='date'  and hasattr(c,'strftime'): return c.strftime('%m/%d')
        if col_fmt=='month': return str(c)[5:]
        return str(c)
    h = (f'<table cellspacing="0" cellpadding="0" '
         f'style="border-collapse:collapse;{FONT}margin:0 0 4px;">'
         f'<thead><tr><th style="{TH}text-align:left;">Category</th>')
    for c in pvt.columns: h += f'<th style="{TH}">{fmt(c)}</th>'
    h += '</tr></thead><tbody>'
    for i, cat in enumerate(pvt.index):
        is_pct  = cat == '% Attrition'
        is_term = cat == 'Terminated'
        bg  = '#FFF3CD' if is_term else (FUNC_BG if is_pct else (ROW_EVEN if i%2==0 else WHITE))
        fw  = 'bold' if (is_pct or is_term) else 'normal'
        td  = f'padding:3px 4px;border:1px solid #D5D8DC;background:{bg};text-align:center;{FONT}font-weight:{fw};'
        h  += f'<tr><td style="{td}text-align:left;padding:3px 7px;">{cat}</td>'
        for c in pvt.columns:
            v = pvt.loc[cat, c]
            val = f'{v:.1f}%' if is_pct else (int(v) if pd.notna(v) and v != 0 else '')
            h += f'<td style="{td}">{val}</td>'
        h += '</tr>'
    return h + '</tbody></table>'

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7: Compute attrition pivots
# ══════════════════════════════════════════════════════════════════════════════
print(f'\nClosed weeks (Mon start): {sorted(EIGHT_WEEKS)}')

# Table 4: last 8 closed weeks, column label = Monday start of week
pvt_8wk = _pivot_attrition(
    df_hc_all,
    col_fn=to_week_monday,
    filter_periods=EIGHT_WEEKS
)

# Table 5: monthly Jan → current month, last-day snapshot per OracleID
pvt_monthly_raw = _pivot_attrition(
    df_hc_all,
    col_fn=lambda x: x.strftime('%Y-%m') if pd.notna(x) else None
)
if not pvt_monthly_raw.empty:
    cur_ym      = dt.strftime('%Y-%m')
    month_cols  = [c for c in pvt_monthly_raw.columns
                   if str(c).startswith(str(dt.year)) and str(c) <= cur_ym]
    pvt_monthly = pvt_monthly_raw[month_cols] if month_cols else pvt_monthly_raw
else:
    pvt_monthly = pvt_monthly_raw

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8: Build all HTML tables + email
# ══════════════════════════════════════════════════════════════════════════════
hc_pivot_html    = build_hc_pivot_html(df_hc, lob3_col)
ews_total_html   = build_ews_total_html(df_ews_out, 'Supervisor Name', 'LOB', 'IEX ID')
ews_list_html    = build_ews_list_html(df_ews_out)
atr_8wk_html     = build_attrition_html(pvt_8wk,    col_fmt='date')
atr_monthly_html = build_attrition_html(pvt_monthly, col_fmt='month')

FONT_S  = f'{FONT}color:#222;'
TITLE_S = f'margin:14px 0 4px;{FONT}font-size:12px;color:{D_DARK};font-weight:bold;'

email_body = f"""<div style="padding:20px 24px;background:#ffffff;{FONT_S}">
  <p>Dear team,</p>
  <p>Please find the <strong>Expedia VN HC {WB_LABEL}</strong> as of <strong>{DATE_DISPLAY}</strong> as attached.</p>
  <p style="{TITLE_S}">1. HC Summary &mdash; {WB_LABEL}</p>
  {hc_pivot_html}
  <p style="{TITLE_S}">2. EWS Summary by Supervisor</p>
  {ews_total_html}
  <p style="{TITLE_S}">3. EWS Cases &mdash; From {TARGET_DATE} (sorted by LWD)</p>
  {ews_list_html}
  <p style="{TITLE_S}">4. HC Summary WoW &mdash; Last 8 Weeks</p>
  {atr_8wk_html}
  <p style="{TITLE_S}">5. HC Summary MoM &mdash; Monthly {dt.year}</p>
  {atr_monthly_html}
  <p style="margin-top:20px;line-height:1.6;{FONT_S}">
    Thanks &amp; Regards,<br><strong>Chinh Nguyen</strong><br>
    Analyst, WFM Real Time Management<br>
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam<br>
    Ph No: +84 986 473 419 | Email:
    <a href="mailto:huuchinh.nguyen@concentrix.com">huuchinh.nguyen@concentrix.com</a>
  </p>
</div>"""

display(HTML(email_body))
print(f'\nFile 1: {file1_name} | File 2: {file2_name} | EWS: {len(df_ews_out)}')

if SEND_EMAIL:
    import pythoncom, psutil, win32com.client
    def send_email_outlook(to, cc, subject, body, attachments=None, quit_after=False):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower()=='outlook.exe' for p in psutil.process_iter(['name']))
        if not was_on:
            for exe in [r'C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE',
                        r'C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE']:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject('Outlook.Application'); break
                except: pass
        try:
            ol = win32com.client.Dispatch('Outlook.Application')
            ol.GetNamespace('MAPI').Logon()
            mail = ol.CreateItem(0)
            mail.To=to; mail.CC=cc; mail.Subject=subject; mail.HTMLBody=body
            for att in (attachments or []):
                if att and os.path.exists(os.path.abspath(att)):
                    mail.Attachments.Add(os.path.abspath(att))
                    print(f'Attached: {os.path.basename(att)}')
            mail.Send(); print(f'Email sent to: {to}'); time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit()
                except: pass
    atts = [file1_path]
    if file2_path: atts.append(file2_path)
    send_email_outlook(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, email_body, atts)

shutil.rmtree(TEMP_DIR, ignore_errors=True)
print('Done.')


Loading HC Extend CSVs...
  Combined: 292346 rows
  Attrition base: 292346 rows | LOB col: LOB_3
  LOB unique vals: ['-', 'HPO', 'HPO Training', 'LG Chat', 'LG Chat CSG', 'LG Chat CSG Training', 'LG Nesting Chat', 'Lodging_Training', 'NL Chat', 'Support', 'Support - LG', 'Support - NL', 'Support_Flex_QA', 'Support_Flex_SME', 'Support_Flex_Trainer', 'Support_LG_Nesting', 'Support_LG_Task', 'Support_NL_Nesting', 'TQA', 'WFM']
  Target week rows: 130
Loading EWS...
  EWS display: 9 rows
Loading WD current month...
  WD current: WD_2026_08.xlsx
File 1: HC Master Database - WB260824.xlsx | File 2: WD_2026_08.xlsx

Loading all WD Termination sheets...
  WD Terminated unique: 641 | date=_eff | emp=None

Closed weeks (Mon start): [datetime.date(2026, 7, 6), datetime.date(2026, 7, 13), datetime.date(2026, 7, 20), datetime.date(2026, 7, 27), datetime.date(2026, 8, 3), datetime.date(2026, 8, 10), datetime.date(2026, 8, 17), datetime.date(2026, 8, 24)]



File 1: HC Master Database - WB260824.xlsx | File 2: WD_2026_08.xlsx | EWS: 9
Done.
